# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

* **Task Type:** Learning to Rank (LTR) / Scoring Pipeline.
* **Explanation:** We are assigning an opportunity/decay score to each URL and sorting them into a prioritized queue. This allows the editorial team to consume the list from top to bottom, addressing the highest-value, highest-decay pages first.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

* **What we want to predict (The Target):** The actual drop in organic impressions or clicks over a forward-looking window (e.g., next 30 days) relative to a historical baseline.
* **The Proxy:** Since we cannot perfectly predict the future, our proxy will be a composite calculation: a sustained downward trend in Google Search Console (GSC) clicks/impressions combined with a drop in historical click-through rate (CTR) while overall query search volume remains stable or increasing.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

* **Offline ML Metric:** Precision at $K$ ($P@K$, where $K=50$ or $100$) and Normalized Discounted Cumulative Gain (NDCG). We care deeply that the top $K$ pages pushed to the writers are truly the ones suffering from severe, recoverable decay.
* **Business Success Metric:** The lift in organic traffic (clicks) captured by refreshed pages compared to an un-refreshed control group over a 60-day post-update window, maximizing the "Return on Editorial Effort."

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import pandas as pd
import numpy as np

# Load the dataset slice
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Clean up average position tracking issues (replace 0 with NaN)
df['clean_position'] = df['avg_position'].replace(0, np.nan)

# 4. The unit of analysis
# One row = One unique Content ID / Performance snapshot
print("--- UNIT OF ANALYSIS DATAFRAME (One row = One unique content piece) ---")
display(df[['content_id', 'clicks_last_30d', 'impressions_last_30d', 'ctr', 'clean_position', 'trend_direction']].head())

# Sketch the target column (Binary indicator of decay for training)
# Let's construct a proxy target where 1 = High Priority Decay, 0 = Stable/Healthy
df['target_is_decayed'] = ((df['trend_direction'] == 'down') & (df['ctr'] < df['ctr'].median())).astype(int)

print("\n--- TARGET COLUMN SKETCH (Distribution of our proxy target) ---")
print(df['target_is_decayed'].value_counts(normalize=True).rename({0: "Healthy (0)", 1: "Decaying/Actionable (1)"}))

--- UNIT OF ANALYSIS DATAFRAME (One row = One unique content piece) ---


,content_id,clicks_last_30d,impressions_last_30d,ctr,clean_position,trend_direction
0,content_304f48230142,2,578,0.76,10.6,down
1,content_a1fb4e703a9e,2,2501,0.05,20.3,down
2,content_9aa793d4d895,1,2382,0.09,36.5,down
3,content_331d6c4de07b,22,3626,0.49,6.2,stable
4,content_d99b7a2d90ca,10,4211,0.13,44.0,down



--- TARGET COLUMN SKETCH (Distribution of our proxy target) ---
target_is_decayed
Healthy (0)                0.743433
Decaying/Actionable (1)    0.256567
Name: proportion, dtype: float64


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed heuristic rule (like *"Flag any page if clicks drop by 20%"*) fails in production for three reasons:
1. **Seasonality & Market Shifts:** A 20% drop might simply reflect a market-wide seasonal slump, not actual content decay. A hard rule would trigger massive false positives, wasting copywriter time.
2. **Volatile Baselines:** High-traffic pillar pages fluctuate differently than long-tail niche pages. A single threshold cannot scale across thousands of highly varied URLs.
3. **Complex Interaction of Features:** Content decay is signaled by the *tangled relationship* between position drops, CTR decay, and impressions. Machine learning can map these non-linear interactions dynamically across different domains where strict static logic breaks.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.